<a href="https://colab.research.google.com/github/saketh-varma06/DL_ASSIGNMENTS/blob/main/DL_ASS_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning Assignment: Text Classification on AG News using BERT
This notebook demonstrates a complete end-to-end PyTorch pipeline to:
1. Download and preprocess the AG News dataset.
2. Implement a BERT-based sequence classification model in PyTorch.
3. Train and fine-tune the model with appropriate learning rate schedulers and optimizations.
4. Evaluate the model with metrics (Accuracy, Precision, Recall, F1-score).
5. Plot training/validation loss curves.
6. Compare the results against a baseline model.

In [ ]:
!pip install -q datasets transformers accelerate evaluate

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
import urllib.request
import os

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# 1. Download and preprocess AG News from Raw URLs
print("Downloading AG News CSV files manually...")
train_url = "https://raw.githubusercontent.com/mhjabreel/CharCNN/master/data/ag_news_csv/train.csv"
test_url = "https://raw.githubusercontent.com/mhjabreel/CharCNN/master/data/ag_news_csv/test.csv"

urllib.request.urlretrieve(train_url, "train.csv")
urllib.request.urlretrieve(test_url, "test.csv")

# Load into Pandas
df_train = pd.read_csv("train.csv", header=None, names=["label", "title", "description"])
df_test = pd.read_csv("test.csv", header=None, names=["label", "title", "description"])

# Combine title and description to form the input text, align label to 0-3 instead of 1-4
df_train["text"] = df_train["title"] + ". " + df_train["description"]
df_test["text"] = df_test["title"] + ". " + df_test["description"]
df_train["label"] = df_train["label"] - 1
df_test["label"] = df_test["label"] - 1

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(df_train[["text", "label"]])
test_dataset = Dataset.from_pandas(df_test[["text", "label"]])

# Take a subset to ensure quick training within the Colab environment
train_subset = train_dataset.shuffle(seed=42).select(range(4000))
test_subset = test_dataset.shuffle(seed=42).select(range(1000))

print("Dataset prepared successfully!")
print(train_subset)

HTTPError: HTTP Error 404: Not Found

In [ ]:
# 2. Tokenize and Preprocess with DistilBERT
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_subset.map(preprocess_function, batched=True)
tokenized_test = test_subset.map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

NameError: name 'train_subset' is not defined

In [ ]:
# 3. Define the PyTorch Model & Trainer Configuration
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=4)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# 4. Evaluate and Extract Metrics
eval_results = trainer.evaluate()
print("BERT Fine-Tuning Evaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

NameError: name 'trainer' is not defined

In [ ]:
# 5. Visualize Loss Curve
history = trainer.state.log_history
train_loss = [x['loss'] for x in history if 'loss' in x]
steps = [x['step'] for x in history if 'loss' in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, train_loss, label='Training Loss', color='b')
plt.title('Training Loss Curve')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

NameError: name 'trainer' is not defined

In [ ]:
# 6. Baseline Model: TF-IDF + Logistic Regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("Training TF-IDF + Logistic Regression Baseline...")
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_subset["text"])
X_test = vectorizer.transform(test_subset["text"])
y_train = train_subset["label"]
y_test = test_subset["label"]

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)

baseline_acc = accuracy_score(y_test, y_pred)
print(f"Baseline Model Accuracy: {baseline_acc:.4f}")

Training TF-IDF + Logistic Regression Baseline...


NameError: name 'train_subset' is not defined

In [ ]:
import os
import torch
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Setup seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# 2. Extract and Load User-Specified AG News ZIP Archive
zip_path = "/content/archive (26).zip"
extract_path = "./ag_news_extracted"

print(f"Extracting dataset from {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Search for CSV files inside the extracted directory structure
csv_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("Extracted CSV files:", csv_files)

# Match train and test datasets
train_csv = [f for f in csv_files if "train" in f.lower()][0]
test_csv = [f for f in csv_files if "test" in f.lower()][0]

# Load datasets
df_train = pd.read_csv(train_csv)
df_test = pd.read_csv(test_csv)

# Dynamically identify text and label columns based on content
def preprocess_loaded_df(df):
    # Drop completely empty rows or columns
    df = df.dropna(how='all')

    # Try to find a column with text-like structure or the name "text" / "description"
    text_col = None
    label_col = None

    for col in df.columns:
        if "text" in str(col).lower() or "desc" in str(col).lower():
            text_col = col
        elif "label" in str(col).lower():
            label_col = col

    if text_col is None:
        # Default fallback to first non-numeric object column
        object_cols = df.select_dtypes(include=['object']).columns
        text_col = object_cols[0] if len(object_cols) > 0 else df.columns[0]
    if label_col is None:
        # Default fallback to numeric or last column
        label_col = df.columns[-1]

    # Clean and cast the columns
    df = df.rename(columns={text_col: "text", label_col: "label"})
    df = df[df["text"].notna() & df["label"].notna()]

    # Safely convert labels to integers (ignoring headers or string labels gracefully)
    df["label"] = pd.to_numeric(df["label"], errors='coerce')
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)

    return df[["text", "label"]].copy()

df_train = preprocess_loaded_df(df_train)
df_test = preprocess_loaded_df(df_test)

# Adjust label offsets if minimum label starts at 1 (to make it 0-indexed for PyTorch loss)
min_label = min(df_train["label"].min(), df_test["label"].min())
if min_label == 1:
    df_train["label"] = df_train["label"] - 1
    df_test["label"] = df_test["label"] - 1

num_classes = len(df_train["label"].unique())
print(f"Found {num_classes} unique target classes.")

# Subsample to ensure fast execution in Colab
train_data = Dataset.from_pandas(df_train).shuffle(seed=42).select(range(min(1000, len(df_train))))
test_data = Dataset.from_pandas(df_test).shuffle(seed=42).select(range(min(200, len(df_test))))

# 3. Tokenization
print("Tokenizing with DistilBERT model checkpoint...")
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_data.map(tokenize_fn, batched=True)
tokenized_test = test_data.map(tokenize_fn, batched=True)

# 4. Define Model architecture
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_classes)

# 5. Fine-Tuning Setup
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Training the Deep Learning Model...")
trainer.train()

# 6. Evaluation
print("Evaluating PyTorch Model...")
eval_results = trainer.evaluate()
print(f"DistilBERT Model Validation Accuracy: {eval_results['eval_accuracy']:.4f}")

# 7. Visualization of Training History
history = trainer.state.log_history
train_loss = [x['loss'] for x in history if 'loss' in x]
steps = [x['step'] for x in history if 'loss' in x]

plt.figure(figsize=(10, 4))
plt.plot(steps, train_loss, marker='o', label='Training Loss', color='darkblue')
plt.title('BERT Training Loss Curve')
plt.xlabel('Optimization Steps')
plt.ylabel('Loss Value')
plt.grid(True)
plt.legend()
plt.show()

# 8. Baseline Model: TF-IDF + Logistic Regression
print("Training Baseline Model...")
vectorizer = TfidfVectorizer(max_features=1000)
X_train = vectorizer.fit_transform(train_data["text"])
X_test = vectorizer.transform(test_data["text"])
y_train = train_data["label"]
y_test = test_data["label"]

baseline = LogisticRegression(max_iter=500)
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred)

print("\n--- Performance Comparison Summary ---")
print(f"[Baseline] TF-IDF + Logistic Regression Accuracy: {baseline_acc:.4f}")
print(f"[Improved] Fine-tuned DistilBERT PyTorch Accuracy  : {eval_results['eval_accuracy']:.4f}")
print(f"Net performance delta                        : {eval_results['eval_accuracy'] - baseline_acc:+.4f}")

Extracting dataset from /content/archive (26).zip...
Extracted CSV files: ['./ag_news_extracted/training.csv', './ag_news_extracted/test.csv', './ag_news_extracted/validation.csv']
Found 6 unique target classes.
Tokenizing with DistilBERT model checkpoint...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'